# FAISS functional walkthrough

This notebook is a self-contained, engineer-friendly walkthrough of the FAISS functional-test flow.
It mirrors the logic in:

- `tests/functional/conftest.py`
- `tests/functional/data.py`
- `tests/functional/filter_assertions.py`
- `tests/functional/test_faiss_filters.py`

## Prerequisites

- Run the notebook from this repository checkout.
- Docker Engine and `docker compose` must be available.
- The host ports used by the FAISS test stack must be free: `6104` for `vector-retriever` and `9714` for the embedding service.
- The notebook assumes the kernel can import `requests`.

## What this notebook demonstrates

1. Configure the FAISS backend with the same ports and request shapes used by the functional tests.
2. Start the docker-compose stack and wait for `vector-retriever` readiness.
3. Seed deterministic fixture documents into the backend.
4. Preserve the FAISS-specific save/restart behavior from the tests so the running service reloads the seeded index from disk.
5. Exercise `/health`, `/ready`, `/capabilities/filters`, and `/query`.
6. Walk through batch queries, `explain_filters`, `top_k`, and every filter case in `FILTER_CASES`.
7. Tear the stack down cleanly when exploration is finished.


## Imports and fixture constants

Load standard-library helpers, notebook display utilities, HTTP tooling, and the shared deterministic fixture constants from the functional-test package.


In [1]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
import uuid
from copy import deepcopy
from pathlib import Path
from typing import Any

import requests
from IPython.display import JSON, Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / "pyproject.toml").exists() and (path / "docker").exists():
            return path
    raise FileNotFoundError("Could not find the vector-retriever repository root from the current notebook kernel.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tests.functional.data import DOCUMENTS, FILTER_CASES

print(f"Repository root: {REPO_ROOT}")
print(f"Loaded {len(DOCUMENTS)} fixture documents and {len(FILTER_CASES)} filter cases.")


Repository root: /home/intel/workbench/integration/lib/microservices/vector-retriever/vector-retriever
Loaded 6 fixture documents and 20 filter cases.


## FAISS-specific environment and port settings

These values mirror the FAISS entries in `tests/functional/conftest.py`, while using a notebook-specific compose project name and a container-local path for the persisted FAISS index (so save/reload works across container restarts).

In [2]:
BACKEND = "faiss"
PORT_MAP = {
    "VECTOR_RETRIEVER_HOST_PORT": "6104",
    "EMBEDDING_SERVER_PORT": "9714",
}
DEFAULT_TEST_EMBEDDING_MODEL_NAME = "CLIP/clip-vit-b-32"

RUN_ID = uuid.uuid4().hex[:8]
COMPOSE_PROJECT_NAME = f"vrfaissnb{RUN_ID}"
COMPOSE_FILES = ["docker/compose.yaml", "docker/compose.faiss.yaml"]

# Keep a small host-side runtime folder for notebook artifacts/logging checks.
NOTEBOOK_RUNTIME_DIR = REPO_ROOT / "tests" / "functional" / "notebooks" / ".runtime" / f"faiss-functional-{RUN_ID}"
# FAISS save/load happens inside the vector-retriever container, so this path must be container-local.
FAISS_INDEX_PATH = Path(f"/tmp/vector-retriever/faiss-functional-{RUN_ID}/faiss-index")
BASE_URL = f"http://localhost:{PORT_MAP['VECTOR_RETRIEVER_HOST_PORT']}"

OV_MODELS_SOURCE_VOLUME = "docker_ov_models"
OV_MODELS_DEST_VOLUME = f"{COMPOSE_PROJECT_NAME}_ov-models"

STACK_ENV = None
NOTEBOOK_STATE = {
    "stack_started": False,
    "seeded": False,
}

display(
    JSON(
        {
            "backend": BACKEND,
            "base_url": BASE_URL,
            "compose_project_name": COMPOSE_PROJECT_NAME,
            "compose_files": COMPOSE_FILES,
            "ports": PORT_MAP,
            "faiss_index_path": str(FAISS_INDEX_PATH),
        },
        expanded=True,
    )
)

<IPython.core.display.JSON object>

## Local helper functions

Everything below is defined directly in the notebook so the walkthrough stays self-contained.
The helpers intentionally mirror the functional-test flow: docker-compose execution, OV-model volume preparation, readiness polling, deterministic seeding, API calls, result display, and cleanup.


In [3]:
FILTER_CASES_BY_NAME = {case["name"]: deepcopy(case) for case in FILTER_CASES}


def require_docker() -> None:
    if shutil.which("docker") is None:
        raise EnvironmentError("docker CLI is required to run this notebook.")


def build_env() -> dict[str, str]:
    env = os.environ.copy()
    env.update(PORT_MAP)
    env["COMPOSE_PROJECT_NAME"] = COMPOSE_PROJECT_NAME
    env["RETRIEVER_BACKEND"] = BACKEND

    index_name = f"vr_functional_{BACKEND}_{RUN_ID}"
    env["INDEX_NAME"] = index_name
    env["VS_INDEX_NAME"] = index_name
    env["EMBEDDING_MODEL_NAME"] = env.get("EMBEDDING_MODEL_NAME", "").strip() or DEFAULT_TEST_EMBEDDING_MODEL_NAME
    env["EMBEDDING_DEVICE"] = "CPU"
    env["EMBEDDING_USE_OV"] = "true"
    env["FAISS_INDEX_PATH"] = str(FAISS_INDEX_PATH)
    env["VECTOR_RETRIEVER_LOG_LEVEL"] = "debug"
    return env


def run_compose(args: list[str], env: dict[str, str] | None = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    command = ["docker", "compose"]
    for compose_file in COMPOSE_FILES:
        command.extend(["-f", compose_file])
    command.extend(args)

    print("$", " ".join(command))
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.stderr.strip():
        print(completed.stderr.strip())
    if check and completed.returncode != 0:
        raise RuntimeError(
            f"""docker compose failed with exit code {completed.returncode}
stdout:
{completed.stdout}
stderr:
{completed.stderr}"""
        )
    return completed


def ensure_ov_models_volume() -> None:
    if subprocess.run(["docker", "volume", "inspect", OV_MODELS_DEST_VOLUME], capture_output=True).returncode == 0:
        print(f"OV models destination volume already exists: {OV_MODELS_DEST_VOLUME}")
        return

    if subprocess.run(["docker", "volume", "inspect", OV_MODELS_SOURCE_VOLUME], capture_output=True).returncode != 0:
        print(
            "Cached OV models source volume is not present; the embedding service will populate models at runtime."
        )
        return

    subprocess.run(["docker", "volume", "create", OV_MODELS_DEST_VOLUME], check=True, capture_output=True)
    subprocess.run(
        [
            "docker",
            "run",
            "--rm",
            "-v",
            f"{OV_MODELS_SOURCE_VOLUME}:/src",
            "-v",
            f"{OV_MODELS_DEST_VOLUME}:/dst",
            "alpine",
            "sh",
            "-c",
            "cp -r /src/. /dst/",
        ],
        check=True,
        capture_output=True,
    )
    print(f"Copied cached OV models into {OV_MODELS_DEST_VOLUME}")


def wait_for_ready(base_url: str = BASE_URL, timeout_seconds: int = 600) -> dict[str, Any]:
    start = time.time()
    last_error = ""
    while time.time() - start < timeout_seconds:
        try:
            response = requests.get(f"{base_url}/ready", timeout=10)
            if response.status_code == 200:
                payload = response.json()
                if payload.get("status") == "ready":
                    print(f"Service is ready at {base_url}")
                    return payload
                last_error = f"unexpected /ready payload: {payload}"
            else:
                last_error = f"/ready status={response.status_code} body={response.text}"
        except Exception as exc:  # noqa: BLE001
            last_error = str(exc)
        time.sleep(3)
    raise AssertionError(f"vector-retriever did not become ready: {last_error}")


def api_get(path: str, *, params: dict[str, Any] | None = None, timeout: int = 15) -> requests.Response:
    response = requests.get(f"{BASE_URL}{path}", params=params, timeout=timeout)
    print(f"GET {path} -> {response.status_code}")
    return response


def api_post(path: str, payload: Any, *, timeout: int = 60) -> requests.Response:
    response = requests.post(f"{BASE_URL}{path}", json=payload, timeout=timeout)
    print(f"POST {path} -> {response.status_code}")
    return response


def response_json(response: requests.Response) -> Any:
    try:
        payload = response.json()
    except ValueError as exc:
        raise AssertionError(f"Expected JSON response but received: {response.text}") from exc
    if response.status_code >= 400:
        raise AssertionError(json.dumps(payload, indent=2) if isinstance(payload, dict) else payload)
    return payload


def show_json(payload: Any, *, expanded: bool = False) -> None:
    display(JSON(payload, expanded=expanded))


def result_rows(result: dict[str, Any]) -> list[dict[str, Any]]:
    rows = []
    for item in result.get("items", []):
        metadata = item.get("metadata", {})
        rows.append(
            {
                "video_id": metadata.get("video_id"),
                "camera_zone": metadata.get("camera_zone"),
                "frame_number": metadata.get("frame_number"),
                "bucket_name": metadata.get("bucket_name"),
                "score": round(float(item.get("score", 0.0)), 6),
            }
        )
    return sorted(rows, key=lambda row: (row["video_id"] or ""))


def show_result_block(result: dict[str, Any], *, header: str | None = None) -> None:
    matched_ids = [row["video_id"] for row in result_rows(result)]
    summary = {
        "query_id": result.get("query_id"),
        "query": result.get("query"),
        "count": result.get("count"),
        "matched_video_ids": matched_ids,
        "applied_filters": result.get("applied_filters"),
    }
    if header:
        display(Markdown(f"### {header}"))
    show_json(summary, expanded=False)
    if result.get("items"):
        show_json(result_rows(result), expanded=False)


def query_batch(payloads: list[dict[str, Any]]) -> dict[str, Any]:
    response = api_post("/query", payloads, timeout=60)
    body = response_json(response)
    if body.get("errors"):
        raise AssertionError(body["errors"])
    return body


def query_once(payload: dict[str, Any], *, header: str | None = None) -> tuple[dict[str, Any], dict[str, Any]]:
    body = query_batch([payload])
    result = body["results"][0]
    show_result_block(result, header=header)
    return body, result


def seed_backend_data(env: dict[str, str]) -> None:
    backend = env["RETRIEVER_BACKEND"]
    script_lines = [
        "import json",
        "from src.common.settings import settings",
        "from src.retriever.backend_factory import get_vectordb",
        f"expected_backend = {backend!r}",
        "assert settings.RETRIEVER_BACKEND == expected_backend, (",
        "    f'backend mismatch: expected {expected_backend!r}, got {settings.RETRIEVER_BACKEND!r}'",
        ")",
        f"docs = json.loads({json.dumps(DOCUMENTS)!r})",
        "store = get_vectordb()",
        "texts = [item['page_content'] for item in docs]",
        "metadatas = [item['metadata'] for item in docs]",
        "ids = [item['metadata']['video_id'] for item in docs]",
        "try:",
        "    store.add_texts(texts=texts, metadatas=metadatas, ids=ids)",
        "except TypeError:",
        "    store.add_texts(texts=texts, metadatas=metadatas)",
    ]
    if backend == "faiss":
        script_lines.extend(
            [
                "from pathlib import Path as _Path",
                "_index_path = (settings.FAISS_INDEX_PATH or '').strip()",
                "if _index_path and hasattr(store, 'save_local'):",
                "    _Path(_index_path).mkdir(parents=True, exist_ok=True)",
                "    store.save_local(_index_path)",
                "    print('saved faiss index to', _index_path)",
            ]
        )
    script_lines.append("print('seeded', len(docs))")
    script = chr(10).join(script_lines)

    run_compose(["exec", "-T", "vector-retriever", "python", "-c", script], env=env)


def verify_seed_visible(expected_count: int = len(DOCUMENTS)) -> dict[str, Any]:
    payload = {
        "query_id": "seed-verification",
        "query": "fixture retrieval anchor",
        "top_k": 100,
    }
    _, result = query_once(payload, header="Seed verification")
    if result["count"] < expected_count:
        raise AssertionError(f"expected at least {expected_count} seeded docs, got {result['count']}")
    return result


def get_filter_case(name: str) -> dict[str, Any]:
    try:
        return deepcopy(FILTER_CASES_BY_NAME[name])
    except KeyError as exc:
        raise KeyError(f"Unknown filter case: {name}") from exc


def run_filter_case(case: dict[str, Any]) -> tuple[dict[str, Any], dict[str, Any]]:
    payload = {
        "query_id": case["name"],
        "query": "fixture retrieval anchor",
        "top_k": 100,
        "explain_filters": True,
    }
    payload.update(deepcopy(case["payload"]))
    body, result = query_once(payload, header=f"Filter case: {case['name']}")
    matched_ids = {item["metadata"]["video_id"] for item in result["items"]}
    expected_ids = set(case["expected_ids"])
    if matched_ids != expected_ids:
        raise AssertionError(f"case={case['name']} expected={sorted(expected_ids)} got={sorted(matched_ids)}")
    display(
        Markdown(
            f"""**Expected IDs:** `{sorted(expected_ids)}`  
**Observed IDs:** `{sorted(matched_ids)}`"""
        )
    )
    return body, result


def cleanup_stack(env: dict[str, str] | None = None, *, remove_runtime_dir: bool = True) -> None:
    active_env = env or STACK_ENV
    if active_env is None:
        print("No stack environment has been created in this kernel yet.")
    else:
        run_compose(["down", "-v", "--remove-orphans"], env=active_env, check=False)
    NOTEBOOK_STATE["stack_started"] = False
    NOTEBOOK_STATE["seeded"] = False
    if remove_runtime_dir and NOTEBOOK_RUNTIME_DIR.exists():
        shutil.rmtree(NOTEBOOK_RUNTIME_DIR, ignore_errors=True)
        print(f"Removed runtime directory: {NOTEBOOK_RUNTIME_DIR}")


## Start the FAISS stack with docker compose

This cell mirrors the `docker compose up -d --build` step from the functional fixture setup and prepares the optional OV-model cache volume when available.


In [4]:
require_docker()
STACK_ENV = build_env()
NOTEBOOK_RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

ensure_ov_models_volume()
run_compose(["up", "-d", "--build"], env=STACK_ENV)
run_compose(["ps"], env=STACK_ENV)
NOTEBOOK_STATE["stack_started"] = True
display(Markdown(f"Started the `{BACKEND}` stack at `{BASE_URL}`."))


Copied cached OV models into vrfaissnb6a1e40fb_ov-models
$ docker compose -f docker/compose.yaml -f docker/compose.faiss.yaml up -d --build
#1 [internal] load local bake definitions
#1 reading from stdin 945B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 1.10kB done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.12.13-slim
#3 DONE 2.0s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [1/8] FROM docker.io/library/python:3.12.13-slim@sha256:090ba77e2958f6af52a5341f788b50b032dd4ca28377d2893dcf1ecbdfdfe203
#5 resolve docker.io/library/python:3.12.13-slim@sha256:090ba77e2958f6af52a5341f788b50b032dd4ca28377d2893dcf1ecbdfdfe203 0.0s done
#5 CACHED

#6 [internal] load build context
#6 transferring context: 4.01kB done
#6 DONE 0.0s

#7 [2/8] RUN groupadd --gid 1000 appuser &&     useradd --uid 1000 --gid appuser --shell /bin/bash --create-home appuser
#7 DONE 0.2s

#8 [3/8] WORKDIR 

Started the `faiss` stack at `http://localhost:6104`.

## Wait for readiness

Poll `/ready` until `vector-retriever` reports `status="ready"`.


In [5]:
ready_payload = wait_for_ready()
show_json(ready_payload, expanded=True)


Service is ready at http://localhost:6104


<IPython.core.display.JSON object>

## Seed deterministic fixture documents

This seeds the same `DOCUMENTS` payload used by the functional tests.
For FAISS, the seed step must persist the in-memory index and then restart `vector-retriever` so the service reloads the saved index from disk.


In [6]:
seed_backend_data(STACK_ENV)
NOTEBOOK_STATE["seeded"] = True

run_compose(["restart", "vector-retriever"], env=STACK_ENV)
wait_for_ready()
display(
    Markdown(
        "Seeded the deterministic fixture documents and restarted `vector-retriever` "
        "so the FAISS backend reloads the saved local index."
    )
)


$ docker compose -f docker/compose.yaml -f docker/compose.faiss.yaml exec -T vector-retriever python -c import json
from src.common.settings import settings
from src.retriever.backend_factory import get_vectordb
expected_backend = 'faiss'
assert settings.RETRIEVER_BACKEND == expected_backend, (
    f'backend mismatch: expected {expected_backend!r}, got {settings.RETRIEVER_BACKEND!r}'
)
docs = json.loads('[{"page_content": "red car at intersection", "metadata": {"video_id": "vid-001", "camera_zone": "north", "description": "red car at intersection", "prefix": "alpha-route", "frame_number": 10, "bucket_name": "cam-a", "tags": ["traffic", "vehicle", "red"], "created_at": "2026-03-10T10:00:00+00:00", "optional_note": "doc has note"}}, {"page_content": "blue bus near station", "metadata": {"video_id": "vid-002", "camera_zone": "south", "description": "blue bus near station", "prefix": "beta-route", "frame_number": 20, "bucket_name": "cam-b", "tags": ["traffic", "vehicle", "bus"], "created_a

Seeded the deterministic fixture documents and restarted `vector-retriever` so the FAISS backend reloads the saved local index.

## Verify that the seed is visible

Before exploring filter behavior, confirm that the service can retrieve the seeded documents through `/query`.


In [7]:
seed_verification_result = verify_seed_visible(expected_count=len(DOCUMENTS))
seed_verification_result


POST /query -> 200


### Seed verification

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

{'query_id': 'seed-verification',
 'query': 'fixture retrieval anchor',
 'count': 6,
 'items': [{'score': 1.4777004718780518,
   'metadata': {'video_id': 'vid-001',
    'camera_zone': 'north',
    'description': 'red car at intersection',
    'prefix': 'alpha-route',
    'frame_number': 10,
    'bucket_name': 'cam-a',
    'tags': ['traffic', 'vehicle', 'red'],
    'created_at': '2026-03-10T10:00:00+00:00',
    'optional_note': 'doc has note'},
   'page_content': 'red car at intersection'},
  {'score': 1.4708623886108398,
   'metadata': {'video_id': 'vid-002',
    'camera_zone': 'south',
    'description': 'blue bus near station',
    'prefix': 'beta-route',
    'frame_number': 20,
    'bucket_name': 'cam-b',
    'tags': ['traffic', 'vehicle', 'bus'],
    'created_at': '2026-03-15T12:00:00+00:00'},
   'page_content': 'blue bus near station'},
  {'score': 1.2737712860107422,
   'metadata': {'video_id': 'vid-006',
    'camera_zone': 'south-west',
    'description': 'empty intersection at 

## Endpoint check: `/health`

`/health` is a liveness probe and should return a simple `status="ok"` payload.


In [8]:
health_payload = response_json(api_get("/health"))
show_json(health_payload, expanded=True)
assert health_payload["status"] == "ok"


GET /health -> 200


<IPython.core.display.JSON object>

## Endpoint check: `/ready`

`/ready` should confirm that the FAISS-backed service can initialize its runtime dependencies.


In [9]:
ready_endpoint_payload = response_json(api_get("/ready"))
show_json(ready_endpoint_payload, expanded=True)
assert ready_endpoint_payload["status"] == "ready"


GET /ready -> 200


<IPython.core.display.JSON object>

## Endpoint check: `/capabilities/filters`

This endpoint advertises the active backend and the supported filter grammar exposed to clients.


In [10]:
capabilities_payload = response_json(api_get("/capabilities/filters"))
show_json(capabilities_payload, expanded=False)
assert capabilities_payload["active_backend"] == BACKEND
assert BACKEND in {entry["backend"] for entry in capabilities_payload["backends"]}


GET /capabilities/filters -> 200


<IPython.core.display.JSON object>

## Endpoint check: `/query`

Run a simple unfiltered semantic query to show the seeded collection before diving into the targeted filter scenarios.


In [11]:
query_smoke_payload = {
    "query_id": "query-smoke",
    "query": "fixture retrieval anchor",
    "top_k": 5,
}
query_smoke_body, query_smoke_result = query_once(query_smoke_payload, header="Smoke query")
assert query_smoke_result["count"] >= 5 or len(DOCUMENTS) < 5
query_smoke_body


POST /query -> 200


### Smoke query

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

{'request_id': '83552de6-47a9-4430-910a-70f0302fcb2a',
 'results': [{'query_id': 'query-smoke',
   'query': 'fixture retrieval anchor',
   'count': 5,
   'items': [{'score': 1.4708623886108398,
     'metadata': {'video_id': 'vid-002',
      'camera_zone': 'south',
      'description': 'blue bus near station',
      'prefix': 'beta-route',
      'frame_number': 20,
      'bucket_name': 'cam-b',
      'tags': ['traffic', 'vehicle', 'bus'],
      'created_at': '2026-03-15T12:00:00+00:00'},
     'page_content': 'blue bus near station'},
    {'score': 1.2737712860107422,
     'metadata': {'video_id': 'vid-006',
      'camera_zone': 'south-west',
      'description': 'empty intersection at night',
      'prefix': 'delta-night',
      'frame_number': 60,
      'bucket_name': 'cam-e',
      'tags': ['night', 'traffic'],
      'created_at': '2026-04-05T20:10:00+00:00'},
     'page_content': 'empty intersection at night'},
    {'score': 1.1559438705444336,
     'metadata': {'video_id': 'vid-003'

## Batch query

This mirrors `assert_batch_query` from `tests/functional/filter_assertions.py`: two requests are sent in a single batch and both should resolve independently.


In [12]:
batch_payload = [
    {
        "query_id": "batch-q1",
        "query": "fixture retrieval anchor",
        "where": {"field": "video_id", "op": "eq", "value": "vid-001"},
        "top_k": 10,
    },
    {
        "query_id": "batch-q2",
        "query": "fixture retrieval anchor",
        "where": {"field": "video_id", "op": "eq", "value": "vid-002"},
        "top_k": 10,
    },
]
batch_body = query_batch(batch_payload)
assert len(batch_body["results"]) == 2
ids_q1 = {item["metadata"]["video_id"] for item in batch_body["results"][0]["items"]}
ids_q2 = {item["metadata"]["video_id"] for item in batch_body["results"][1]["items"]}
assert ids_q1 == {"vid-001"}
assert ids_q2 == {"vid-002"}

for block in batch_body["results"]:
    show_result_block(block, header=f"Batch result: {block['query_id']}")

batch_body


POST /query -> 200


### Batch result: batch-q1

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

### Batch result: batch-q2

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

{'request_id': 'fc730262-8599-41dd-99a8-62303b4241db',
 'results': [{'query_id': 'batch-q1',
   'query': 'fixture retrieval anchor',
   'count': 1,
   'items': [{'score': 1.4777004718780518,
     'metadata': {'video_id': 'vid-001',
      'camera_zone': 'north',
      'description': 'red car at intersection',
      'prefix': 'alpha-route',
      'frame_number': 10,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'vehicle', 'red'],
      'created_at': '2026-03-10T10:00:00+00:00',
      'optional_note': 'doc has note'},
     'page_content': 'red car at intersection'}],
   'applied_filters': {'tags': None,
    'time_filter': None,
    'filters': None,
    'normalized_where': {'field': 'video_id',
     'op': 'eq',
     'value': 'vid-001',
     'all': None,
     'any': None,
     'not': None},
    'warnings': None,
    'compiled_backend_filter': None,
    'dropped_or_rewritten_clauses': None}},
  {'query_id': 'batch-q2',
   'query': 'fixture retrieval anchor',
   'count': 1,
   'item

## `explain_filters`

This mirrors `assert_explain_filters`: enabling `explain_filters=True` should surface a backend-native compiled filter description in the response.


In [13]:
explain_payload = {
    "query_id": "explain-test",
    "query": "fixture retrieval anchor",
    "where": {"field": "frame_number", "op": "gte", "value": 50},
    "top_k": 10,
    "explain_filters": True,
}
explain_body, explain_result = query_once(explain_payload, header="Explain filters")
assert explain_result["applied_filters"].get("compiled_backend_filter") is not None
show_json(explain_result["applied_filters"], expanded=True)
explain_body


POST /query -> 200


### Explain filters

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

{'request_id': 'e612c82b-3b7b-426a-bc85-eeb3a4330cff',
 'results': [{'query_id': 'explain-test',
   'query': 'fixture retrieval anchor',
   'count': 2,
   'items': [{'score': 1.2737712860107422,
     'metadata': {'video_id': 'vid-006',
      'camera_zone': 'south-west',
      'description': 'empty intersection at night',
      'prefix': 'delta-night',
      'frame_number': 60,
      'bucket_name': 'cam-e',
      'tags': ['night', 'traffic'],
      'created_at': '2026-04-05T20:10:00+00:00'},
     'page_content': 'empty intersection at night'},
    {'score': 0.9245074391365051,
     'metadata': {'video_id': 'vid-005',
      'camera_zone': 'north-east',
      'description': 'traffic light malfunction',
      'prefix': 'alpha-signal',
      'frame_number': 50,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'signal'],
      'created_at': '2026-04-01T09:45:00+00:00',
      'optional_note': 'signal issue'},
     'page_content': 'traffic light malfunction'}],
   'applied_filters': {'t

## `top_k` limiting

This mirrors `assert_top_k_limiting`: even when more documents are available, the service should cap the returned results to the requested limit.


In [14]:
top_k_payload = {
    "query_id": "topk-limit",
    "query": "fixture retrieval anchor",
    "top_k": 2,
}
top_k_body, top_k_result = query_once(top_k_payload, header="top_k = 2")
assert top_k_result["count"] == 2
assert len(top_k_result["items"]) == 2
top_k_body


POST /query -> 200


### top_k = 2

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

{'request_id': '86c95469-ee3e-4ade-bb68-4d46f47df1db',
 'results': [{'query_id': 'topk-limit',
   'query': 'fixture retrieval anchor',
   'count': 2,
   'items': [{'score': 0.9245074391365051,
     'metadata': {'video_id': 'vid-005',
      'camera_zone': 'north-east',
      'description': 'traffic light malfunction',
      'prefix': 'alpha-signal',
      'frame_number': 50,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'signal'],
      'created_at': '2026-04-01T09:45:00+00:00',
      'optional_note': 'signal issue'},
     'page_content': 'traffic light malfunction'},
    {'score': 0.9070409536361694,
     'metadata': {'video_id': 'vid-004',
      'camera_zone': 'west',
      'description': 'delivery truck unloading',
      'prefix': 'gamma-log',
      'frame_number': 40,
      'bucket_name': 'cam-d',
      'tags': ['logistics', 'vehicle'],
      'created_at': '2026-03-25T14:15:00+00:00'},
     'page_content': 'delivery truck unloading'}],
   'applied_filters': {'tags': None,


## Image query

This demonstrates the image query modality introduced alongside the existing text query.
An image can be provided as `image_base64` or `image_url` instead of a text `query`.
The two modalities are mutually exclusive — providing both returns a `422` validation error.

The test uses a tiny 1×1 red PNG encoded in base64 so it works without network access.

In [15]:
RED_1X1_PNG_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR4nGP4"
    "z8BQDwAEgAF/pooBPQAAAABJRU5ErkJggg=="
)

# --- image_base64 query ---
image_response = api_post(
    "/query",
    [
        {
            "query_id": "image-base64-test",
            "image": {"type": "image_base64", "image_base64": RED_1X1_PNG_B64},
            "top_k": 5,
        }
    ],
)
image_body = response_json(image_response)
assert not image_body["errors"], image_body["errors"]
image_result = image_body["results"][0]
assert image_result["query_id"] == "image-base64-test"
assert image_result["query"] == "[image_base64]", image_result["query"]
assert image_result["count"] >= 1, image_result["count"]
show_json(image_result, expanded=True)

# --- image query with where filter ---
filtered_response = api_post(
    "/query",
    [
        {
            "query_id": "image-filtered-test",
            "image": {"type": "image_base64", "image_base64": RED_1X1_PNG_B64},
            "where": {"field": "video_id", "op": "eq", "value": "vid-001"},
            "top_k": 10,
        }
    ],
)
filtered_body = response_json(filtered_response)
filtered_result = filtered_body["results"][0]
matched_ids = {item["metadata"]["video_id"] for item in filtered_result["items"]}
assert matched_ids <= {"vid-001"}, matched_ids
show_json(filtered_result, expanded=True)

# --- mutual exclusivity: both query and image should fail ---
invalid_response = api_post(
    "/query",
    [{"query": "test", "image": {"type": "image_base64", "image_base64": RED_1X1_PNG_B64}}],
)
assert invalid_response.status_code == 422, invalid_response.status_code
print(f"Mutual exclusivity correctly rejected with {invalid_response.status_code}")


POST /query -> 200


<IPython.core.display.JSON object>

POST /query -> 200


<IPython.core.display.JSON object>

POST /query -> 422
Mutual exclusivity correctly rejected with 422


## Filter matrix walkthrough

The following sections replay every entry from `FILTER_CASES` in order. Each code cell performs exactly one `/query` API call so you can inspect each result in isolation.


            ### Filter case: `op_eq`

            Exact equality on a scalar metadata field.

            This mirrors the simplest `where` clause shape and should only return `vid-001`.

            Expected matching `video_id` values: `['vid-001']`

            Payload under test:

            ```json
            {
  "where": {
    "field": "video_id",
    "op": "eq",
    "value": "vid-001"
  }
}
            ```


In [16]:
case = get_filter_case("op_eq")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: op_eq

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-001']`  
**Observed IDs:** `['vid-001']`

{'request_id': '759da4ce-03f4-4a9a-b457-bf2149803042',
 'results': [{'query_id': 'op_eq',
   'query': 'fixture retrieval anchor',
   'count': 1,
   'items': [{'score': 1.4777004718780518,
     'metadata': {'video_id': 'vid-001',
      'camera_zone': 'north',
      'description': 'red car at intersection',
      'prefix': 'alpha-route',
      'frame_number': 10,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'vehicle', 'red'],
      'created_at': '2026-03-10T10:00:00+00:00',
      'optional_note': 'doc has note'},
     'page_content': 'red car at intersection'}],
   'applied_filters': {'tags': None,
    'time_filter': None,
    'filters': None,
    'normalized_where': {'field': 'video_id',
     'op': 'eq',
     'value': 'vid-001',
     'all': None,
     'any': None,
     'not': None},
    'warnings': None,
    'compiled_backend_filter': {'video_id': {'$eq': 'vid-001'}},
    'dropped_or_rewritten_clauses': []}}],
 'errors': []}

            ### Filter case: `op_in`

            Membership matching with `op="in"`.

            The query targets two camera zones and should return the north and east fixtures.

            Expected matching `video_id` values: `['vid-001', 'vid-003']`

            Payload under test:

            ```json
            {
  "where": {
    "field": "camera_zone",
    "op": "in",
    "value": [
      "north",
      "east"
    ]
  }
}
            ```


In [17]:
case = get_filter_case("op_in")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: op_in

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-001', 'vid-003']`  
**Observed IDs:** `['vid-001', 'vid-003']`

{'request_id': '41d7a8b8-b373-4311-8990-4733add358de',
 'results': [{'query_id': 'op_in',
   'query': 'fixture retrieval anchor',
   'count': 2,
   'items': [{'score': 1.4777004718780518,
     'metadata': {'video_id': 'vid-001',
      'camera_zone': 'north',
      'description': 'red car at intersection',
      'prefix': 'alpha-route',
      'frame_number': 10,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'vehicle', 'red'],
      'created_at': '2026-03-10T10:00:00+00:00',
      'optional_note': 'doc has note'},
     'page_content': 'red car at intersection'},
    {'score': 1.1559438705444336,
     'metadata': {'video_id': 'vid-003',
      'camera_zone': 'east',
      'description': 'person with bicycle crossing',
      'prefix': 'alpha-bike',
      'frame_number': 30,
      'bucket_name': 'cam-c',
      'tags': ['pedestrian', 'bicycle'],
      'created_at': '2026-03-20T08:30:00+00:00',
      'optional_note': 'pedestrian event'},
     'page_content': 'person with bicycle cros

            ### Filter case: `op_contains`

            Substring matching for free-text metadata.

            This checks that FAISS-side filtering keeps only descriptions containing `intersection`.

            Expected matching `video_id` values: `['vid-001', 'vid-006']`

            Payload under test:

            ```json
            {
  "where": {
    "field": "description",
    "op": "contains",
    "value": "intersection"
  }
}
            ```


In [18]:
case = get_filter_case("op_contains")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: op_contains

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-001', 'vid-006']`  
**Observed IDs:** `['vid-001', 'vid-006']`

{'request_id': '11a8a316-8664-428c-9406-a19f1c0d0108',
 'results': [{'query_id': 'op_contains',
   'query': 'fixture retrieval anchor',
   'count': 2,
   'items': [{'score': 1.4777004718780518,
     'metadata': {'video_id': 'vid-001',
      'camera_zone': 'north',
      'description': 'red car at intersection',
      'prefix': 'alpha-route',
      'frame_number': 10,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'vehicle', 'red'],
      'created_at': '2026-03-10T10:00:00+00:00',
      'optional_note': 'doc has note'},
     'page_content': 'red car at intersection'},
    {'score': 1.2737712860107422,
     'metadata': {'video_id': 'vid-006',
      'camera_zone': 'south-west',
      'description': 'empty intersection at night',
      'prefix': 'delta-night',
      'frame_number': 60,
      'bucket_name': 'cam-e',
      'tags': ['night', 'traffic'],
      'created_at': '2026-04-05T20:10:00+00:00'},
     'page_content': 'empty intersection at night'}],
   'applied_filters': {'tags

            ### Filter case: `op_starts_with`

            Prefix matching with `starts_with`.

            The expected matches are the three fixtures whose `prefix` begins with `alpha`.

            Expected matching `video_id` values: `['vid-001', 'vid-003', 'vid-005']`

            Payload under test:

            ```json
            {
  "where": {
    "field": "prefix",
    "op": "starts_with",
    "value": "alpha"
  }
}
            ```


In [19]:
case = get_filter_case("op_starts_with")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: op_starts_with

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-001', 'vid-003', 'vid-005']`  
**Observed IDs:** `['vid-001', 'vid-003', 'vid-005']`

{'request_id': 'b41d7229-5bd2-4360-8a76-8aa34af81515',
 'results': [{'query_id': 'op_starts_with',
   'query': 'fixture retrieval anchor',
   'count': 3,
   'items': [{'score': 1.4777004718780518,
     'metadata': {'video_id': 'vid-001',
      'camera_zone': 'north',
      'description': 'red car at intersection',
      'prefix': 'alpha-route',
      'frame_number': 10,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'vehicle', 'red'],
      'created_at': '2026-03-10T10:00:00+00:00',
      'optional_note': 'doc has note'},
     'page_content': 'red car at intersection'},
    {'score': 1.1559438705444336,
     'metadata': {'video_id': 'vid-003',
      'camera_zone': 'east',
      'description': 'person with bicycle crossing',
      'prefix': 'alpha-bike',
      'frame_number': 30,
      'bucket_name': 'cam-c',
      'tags': ['pedestrian', 'bicycle'],
      'created_at': '2026-03-20T08:30:00+00:00',
      'optional_note': 'pedestrian event'},
     'page_content': 'person with bic

            ### Filter case: `op_gt`

            Strict numeric comparison with `gt`.

            Only frame numbers greater than 40 should remain after filtering.

            Expected matching `video_id` values: `['vid-005', 'vid-006']`

            Payload under test:

            ```json
            {
  "where": {
    "field": "frame_number",
    "op": "gt",
    "value": 40
  }
}
            ```


In [20]:
case = get_filter_case("op_gt")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: op_gt

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-005', 'vid-006']`  
**Observed IDs:** `['vid-005', 'vid-006']`

{'request_id': '599d70aa-f80d-4dce-8f50-89a56c867fb7',
 'results': [{'query_id': 'op_gt',
   'query': 'fixture retrieval anchor',
   'count': 2,
   'items': [{'score': 1.2737712860107422,
     'metadata': {'video_id': 'vid-006',
      'camera_zone': 'south-west',
      'description': 'empty intersection at night',
      'prefix': 'delta-night',
      'frame_number': 60,
      'bucket_name': 'cam-e',
      'tags': ['night', 'traffic'],
      'created_at': '2026-04-05T20:10:00+00:00'},
     'page_content': 'empty intersection at night'},
    {'score': 0.9245074391365051,
     'metadata': {'video_id': 'vid-005',
      'camera_zone': 'north-east',
      'description': 'traffic light malfunction',
      'prefix': 'alpha-signal',
      'frame_number': 50,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'signal'],
      'created_at': '2026-04-01T09:45:00+00:00',
      'optional_note': 'signal issue'},
     'page_content': 'traffic light malfunction'}],
   'applied_filters': {'tags': N

            ### Filter case: `op_gte`

            Inclusive numeric comparison with `gte`.

            This broadens the previous case to include frame number 40 as well.

            Expected matching `video_id` values: `['vid-004', 'vid-005', 'vid-006']`

            Payload under test:

            ```json
            {
  "where": {
    "field": "frame_number",
    "op": "gte",
    "value": 40
  }
}
            ```


In [21]:
case = get_filter_case("op_gte")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: op_gte

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-004', 'vid-005', 'vid-006']`  
**Observed IDs:** `['vid-004', 'vid-005', 'vid-006']`

{'request_id': 'f0a39a52-a5a4-45df-b6a5-ca0be1ab3088',
 'results': [{'query_id': 'op_gte',
   'query': 'fixture retrieval anchor',
   'count': 3,
   'items': [{'score': 1.2737712860107422,
     'metadata': {'video_id': 'vid-006',
      'camera_zone': 'south-west',
      'description': 'empty intersection at night',
      'prefix': 'delta-night',
      'frame_number': 60,
      'bucket_name': 'cam-e',
      'tags': ['night', 'traffic'],
      'created_at': '2026-04-05T20:10:00+00:00'},
     'page_content': 'empty intersection at night'},
    {'score': 0.9245074391365051,
     'metadata': {'video_id': 'vid-005',
      'camera_zone': 'north-east',
      'description': 'traffic light malfunction',
      'prefix': 'alpha-signal',
      'frame_number': 50,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'signal'],
      'created_at': '2026-04-01T09:45:00+00:00',
      'optional_note': 'signal issue'},
     'page_content': 'traffic light malfunction'},
    {'score': 0.9070409536361694

            ### Filter case: `op_lt`

            Strict numeric comparison with `lt`.

            Only the first two fixtures have frame numbers below 30.

            Expected matching `video_id` values: `['vid-001', 'vid-002']`

            Payload under test:

            ```json
            {
  "where": {
    "field": "frame_number",
    "op": "lt",
    "value": 30
  }
}
            ```


In [22]:
case = get_filter_case("op_lt")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: op_lt

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-001', 'vid-002']`  
**Observed IDs:** `['vid-001', 'vid-002']`

{'request_id': 'd33b5381-51b5-48be-bab1-2f1f99465b38',
 'results': [{'query_id': 'op_lt',
   'query': 'fixture retrieval anchor',
   'count': 2,
   'items': [{'score': 1.4777004718780518,
     'metadata': {'video_id': 'vid-001',
      'camera_zone': 'north',
      'description': 'red car at intersection',
      'prefix': 'alpha-route',
      'frame_number': 10,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'vehicle', 'red'],
      'created_at': '2026-03-10T10:00:00+00:00',
      'optional_note': 'doc has note'},
     'page_content': 'red car at intersection'},
    {'score': 1.4708623886108398,
     'metadata': {'video_id': 'vid-002',
      'camera_zone': 'south',
      'description': 'blue bus near station',
      'prefix': 'beta-route',
      'frame_number': 20,
      'bucket_name': 'cam-b',
      'tags': ['traffic', 'vehicle', 'bus'],
      'created_at': '2026-03-15T12:00:00+00:00'},
     'page_content': 'blue bus near station'}],
   'applied_filters': {'tags': None,
    't

            ### Filter case: `op_lte`

            Inclusive numeric comparison with `lte`.

            This adds the frame-30 document to the `lt` result set.

            Expected matching `video_id` values: `['vid-001', 'vid-002', 'vid-003']`

            Payload under test:

            ```json
            {
  "where": {
    "field": "frame_number",
    "op": "lte",
    "value": 30
  }
}
            ```


In [23]:
case = get_filter_case("op_lte")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: op_lte

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-001', 'vid-002', 'vid-003']`  
**Observed IDs:** `['vid-001', 'vid-002', 'vid-003']`

{'request_id': '9e9a5ae4-8147-4d30-b8d0-8ca85e24faf7',
 'results': [{'query_id': 'op_lte',
   'query': 'fixture retrieval anchor',
   'count': 3,
   'items': [{'score': 1.4777004718780518,
     'metadata': {'video_id': 'vid-001',
      'camera_zone': 'north',
      'description': 'red car at intersection',
      'prefix': 'alpha-route',
      'frame_number': 10,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'vehicle', 'red'],
      'created_at': '2026-03-10T10:00:00+00:00',
      'optional_note': 'doc has note'},
     'page_content': 'red car at intersection'},
    {'score': 1.4708623886108398,
     'metadata': {'video_id': 'vid-002',
      'camera_zone': 'south',
      'description': 'blue bus near station',
      'prefix': 'beta-route',
      'frame_number': 20,
      'bucket_name': 'cam-b',
      'tags': ['traffic', 'vehicle', 'bus'],
      'created_at': '2026-03-15T12:00:00+00:00'},
     'page_content': 'blue bus near station'},
    {'score': 1.1559438705444336,
     'met

            ### Filter case: `op_between`

            Range filtering with `between` on a numeric field.

            The expected matches span the inclusive frame-number range 20 through 40.

            Expected matching `video_id` values: `['vid-002', 'vid-003', 'vid-004']`

            Payload under test:

            ```json
            {
  "where": {
    "field": "frame_number",
    "op": "between",
    "value": [
      20,
      40
    ]
  }
}
            ```


In [24]:
case = get_filter_case("op_between")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: op_between

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-002', 'vid-003', 'vid-004']`  
**Observed IDs:** `['vid-002', 'vid-003', 'vid-004']`

{'request_id': 'aeb35f02-8d06-4cd1-9c7b-4d6345a55c82',
 'results': [{'query_id': 'op_between',
   'query': 'fixture retrieval anchor',
   'count': 3,
   'items': [{'score': 1.4708623886108398,
     'metadata': {'video_id': 'vid-002',
      'camera_zone': 'south',
      'description': 'blue bus near station',
      'prefix': 'beta-route',
      'frame_number': 20,
      'bucket_name': 'cam-b',
      'tags': ['traffic', 'vehicle', 'bus'],
      'created_at': '2026-03-15T12:00:00+00:00'},
     'page_content': 'blue bus near station'},
    {'score': 1.1559438705444336,
     'metadata': {'video_id': 'vid-003',
      'camera_zone': 'east',
      'description': 'person with bicycle crossing',
      'prefix': 'alpha-bike',
      'frame_number': 30,
      'bucket_name': 'cam-c',
      'tags': ['pedestrian', 'bicycle'],
      'created_at': '2026-03-20T08:30:00+00:00',
      'optional_note': 'pedestrian event'},
     'page_content': 'person with bicycle crossing'},
    {'score': 0.907040953636169

            ### Filter case: `op_contains_any`

            List membership with `contains_any`.

            A document matches when its `tags` contain at least one requested value.

            Expected matching `video_id` values: `['vid-003', 'vid-005']`

            Payload under test:

            ```json
            {
  "where": {
    "field": "tags",
    "op": "contains_any",
    "value": [
      "bicycle",
      "signal"
    ]
  }
}
            ```


In [25]:
case = get_filter_case("op_contains_any")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: op_contains_any

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-003', 'vid-005']`  
**Observed IDs:** `['vid-003', 'vid-005']`

{'request_id': '24113f49-b360-465f-84e3-39b098216aac',
 'results': [{'query_id': 'op_contains_any',
   'query': 'fixture retrieval anchor',
   'count': 2,
   'items': [{'score': 1.1559438705444336,
     'metadata': {'video_id': 'vid-003',
      'camera_zone': 'east',
      'description': 'person with bicycle crossing',
      'prefix': 'alpha-bike',
      'frame_number': 30,
      'bucket_name': 'cam-c',
      'tags': ['pedestrian', 'bicycle'],
      'created_at': '2026-03-20T08:30:00+00:00',
      'optional_note': 'pedestrian event'},
     'page_content': 'person with bicycle crossing'},
    {'score': 0.9245074391365051,
     'metadata': {'video_id': 'vid-005',
      'camera_zone': 'north-east',
      'description': 'traffic light malfunction',
      'prefix': 'alpha-signal',
      'frame_number': 50,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'signal'],
      'created_at': '2026-04-01T09:45:00+00:00',
      'optional_note': 'signal issue'},
     'page_content': 'traffic l

            ### Filter case: `op_contains_all`

            List membership with `contains_all`.

            Only fixtures that contain both `traffic` and `vehicle` should match.

            Expected matching `video_id` values: `['vid-001', 'vid-002']`

            Payload under test:

            ```json
            {
  "where": {
    "field": "tags",
    "op": "contains_all",
    "value": [
      "traffic",
      "vehicle"
    ]
  }
}
            ```


In [26]:
case = get_filter_case("op_contains_all")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: op_contains_all

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-001', 'vid-002']`  
**Observed IDs:** `['vid-001', 'vid-002']`

{'request_id': 'e609d512-b47e-45dd-9853-b2cbb450b727',
 'results': [{'query_id': 'op_contains_all',
   'query': 'fixture retrieval anchor',
   'count': 2,
   'items': [{'score': 1.4777004718780518,
     'metadata': {'video_id': 'vid-001',
      'camera_zone': 'north',
      'description': 'red car at intersection',
      'prefix': 'alpha-route',
      'frame_number': 10,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'vehicle', 'red'],
      'created_at': '2026-03-10T10:00:00+00:00',
      'optional_note': 'doc has note'},
     'page_content': 'red car at intersection'},
    {'score': 1.4708623886108398,
     'metadata': {'video_id': 'vid-002',
      'camera_zone': 'south',
      'description': 'blue bus near station',
      'prefix': 'beta-route',
      'frame_number': 20,
      'bucket_name': 'cam-b',
      'tags': ['traffic', 'vehicle', 'bus'],
      'created_at': '2026-03-15T12:00:00+00:00'},
     'page_content': 'blue bus near station'}],
   'applied_filters': {'tags': No

            ### Filter case: `op_exists`

            Presence checks for sparse metadata.

            This verifies that documents carrying `optional_note` survive the filter.

            Expected matching `video_id` values: `['vid-001', 'vid-003', 'vid-005']`

            Payload under test:

            ```json
            {
  "where": {
    "field": "optional_note",
    "op": "exists"
  }
}
            ```


In [27]:
case = get_filter_case("op_exists")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: op_exists

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-001', 'vid-003', 'vid-005']`  
**Observed IDs:** `['vid-001', 'vid-003', 'vid-005']`

{'request_id': '74de1d90-849b-40bc-b7e6-63403e5a26fb',
 'results': [{'query_id': 'op_exists',
   'query': 'fixture retrieval anchor',
   'count': 3,
   'items': [{'score': 1.4777004718780518,
     'metadata': {'video_id': 'vid-001',
      'camera_zone': 'north',
      'description': 'red car at intersection',
      'prefix': 'alpha-route',
      'frame_number': 10,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'vehicle', 'red'],
      'created_at': '2026-03-10T10:00:00+00:00',
      'optional_note': 'doc has note'},
     'page_content': 'red car at intersection'},
    {'score': 1.1559438705444336,
     'metadata': {'video_id': 'vid-003',
      'camera_zone': 'east',
      'description': 'person with bicycle crossing',
      'prefix': 'alpha-bike',
      'frame_number': 30,
      'bucket_name': 'cam-c',
      'tags': ['pedestrian', 'bicycle'],
      'created_at': '2026-03-20T08:30:00+00:00',
      'optional_note': 'pedestrian event'},
     'page_content': 'person with bicycle 

            ### Filter case: `op_missing`

            Absence checks for sparse metadata.

            This is the complement of the `op_exists` case.

            Expected matching `video_id` values: `['vid-002', 'vid-004', 'vid-006']`

            Payload under test:

            ```json
            {
  "where": {
    "field": "optional_note",
    "op": "missing"
  }
}
            ```


In [28]:
case = get_filter_case("op_missing")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: op_missing

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-002', 'vid-004', 'vid-006']`  
**Observed IDs:** `['vid-002', 'vid-004', 'vid-006']`

{'request_id': '496554b3-51dd-40e2-80df-dcfdd3b0b7c2',
 'results': [{'query_id': 'op_missing',
   'query': 'fixture retrieval anchor',
   'count': 3,
   'items': [{'score': 1.4708623886108398,
     'metadata': {'video_id': 'vid-002',
      'camera_zone': 'south',
      'description': 'blue bus near station',
      'prefix': 'beta-route',
      'frame_number': 20,
      'bucket_name': 'cam-b',
      'tags': ['traffic', 'vehicle', 'bus'],
      'created_at': '2026-03-15T12:00:00+00:00'},
     'page_content': 'blue bus near station'},
    {'score': 1.2737712860107422,
     'metadata': {'video_id': 'vid-006',
      'camera_zone': 'south-west',
      'description': 'empty intersection at night',
      'prefix': 'delta-night',
      'frame_number': 60,
      'bucket_name': 'cam-e',
      'tags': ['night', 'traffic'],
      'created_at': '2026-04-05T20:10:00+00:00'},
     'page_content': 'empty intersection at night'},
    {'score': 0.9070409536361694,
     'metadata': {'video_id': 'vid-004',

            ### Filter case: `logical_compound`

            Nested logical composition with `all`, `any`, and `not`.

            The request mirrors the most complex structured filter in the parity matrix.

            Expected matching `video_id` values: `['vid-003', 'vid-004']`

            Payload under test:

            ```json
            {
  "where": {
    "all": [
      {
        "field": "frame_number",
        "op": "gte",
        "value": 20
      },
      {
        "any": [
          {
            "field": "camera_zone",
            "op": "eq",
            "value": "east"
          },
          {
            "field": "camera_zone",
            "op": "eq",
            "value": "west"
          }
        ]
      },
      {
        "not": {
          "field": "tags",
          "op": "contains_any",
          "value": [
            "night"
          ]
        }
      }
    ]
  }
}
            ```


In [29]:
case = get_filter_case("logical_compound")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: logical_compound

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-003', 'vid-004']`  
**Observed IDs:** `['vid-003', 'vid-004']`

{'request_id': '5d9560d9-4006-4e69-9567-42bc284a15e1',
 'results': [{'query_id': 'logical_compound',
   'query': 'fixture retrieval anchor',
   'count': 2,
   'items': [{'score': 1.1559438705444336,
     'metadata': {'video_id': 'vid-003',
      'camera_zone': 'east',
      'description': 'person with bicycle crossing',
      'prefix': 'alpha-bike',
      'frame_number': 30,
      'bucket_name': 'cam-c',
      'tags': ['pedestrian', 'bicycle'],
      'created_at': '2026-03-20T08:30:00+00:00',
      'optional_note': 'pedestrian event'},
     'page_content': 'person with bicycle crossing'},
    {'score': 0.9070409536361694,
     'metadata': {'video_id': 'vid-004',
      'camera_zone': 'west',
      'description': 'delivery truck unloading',
      'prefix': 'gamma-log',
      'frame_number': 40,
      'bucket_name': 'cam-d',
      'tags': ['logistics', 'vehicle'],
      'created_at': '2026-03-25T14:15:00+00:00'},
     'page_content': 'delivery truck unloading'}],
   'applied_filters': {'t

            ### Filter case: `legacy_tags_and_time_filter`

            Backward-compatible legacy fields: `tags` plus `time_filter`.

            This confirms that the legacy request shape still normalizes into the active filter path.

            Expected matching `video_id` values: `['vid-002', 'vid-005']`

            Payload under test:

            ```json
            {
  "tags": [
    "traffic"
  ],
  "time_filter": {
    "end": "2026-04-02T00:00:00+00:00",
    "start": "2026-03-12T00:00:00+00:00"
  }
}
            ```


In [30]:
case = get_filter_case("legacy_tags_and_time_filter")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: legacy_tags_and_time_filter

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-002', 'vid-005']`  
**Observed IDs:** `['vid-002', 'vid-005']`

{'request_id': 'e0798a65-3f81-4510-9034-aff339cf09a7',
 'results': [{'query_id': 'legacy_tags_and_time_filter',
   'query': 'fixture retrieval anchor',
   'count': 2,
   'items': [{'score': 1.4708623886108398,
     'metadata': {'video_id': 'vid-002',
      'camera_zone': 'south',
      'description': 'blue bus near station',
      'prefix': 'beta-route',
      'frame_number': 20,
      'bucket_name': 'cam-b',
      'tags': ['traffic', 'vehicle', 'bus'],
      'created_at': '2026-03-15T12:00:00+00:00'},
     'page_content': 'blue bus near station'},
    {'score': 0.9245074391365051,
     'metadata': {'video_id': 'vid-005',
      'camera_zone': 'north-east',
      'description': 'traffic light malfunction',
      'prefix': 'alpha-signal',
      'frame_number': 50,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'signal'],
      'created_at': '2026-04-01T09:45:00+00:00',
      'optional_note': 'signal issue'},
     'page_content': 'traffic light malfunction'}],
   'applied_filters

            ### Filter case: `legacy_filters_map`

            Backward-compatible `filters` map syntax.

            The notebook shows how older clients can still target multiple fields without `where`.

            Expected matching `video_id` values: `['vid-001', 'vid-002']`

            Payload under test:

            ```json
            {
  "filters": {
    "bucket_name": {
      "op": "in",
      "value": [
        "cam-a",
        "cam-b"
      ]
    },
    "frame_number": {
      "op": "between",
      "value": [
        10,
        20
      ]
    }
  }
}
            ```


In [31]:
case = get_filter_case("legacy_filters_map")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: legacy_filters_map

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-001', 'vid-002']`  
**Observed IDs:** `['vid-001', 'vid-002']`

{'request_id': 'bdac2f2f-afcd-445b-b416-32ef3afe3861',
 'results': [{'query_id': 'legacy_filters_map',
   'query': 'fixture retrieval anchor',
   'count': 2,
   'items': [{'score': 1.4777004718780518,
     'metadata': {'video_id': 'vid-001',
      'camera_zone': 'north',
      'description': 'red car at intersection',
      'prefix': 'alpha-route',
      'frame_number': 10,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'vehicle', 'red'],
      'created_at': '2026-03-10T10:00:00+00:00',
      'optional_note': 'doc has note'},
     'page_content': 'red car at intersection'},
    {'score': 1.4708623886108398,
     'metadata': {'video_id': 'vid-002',
      'camera_zone': 'south',
      'description': 'blue bus near station',
      'prefix': 'beta-route',
      'frame_number': 20,
      'bucket_name': 'cam-b',
      'tags': ['traffic', 'vehicle', 'bus'],
      'created_at': '2026-03-15T12:00:00+00:00'},
     'page_content': 'blue bus near station'}],
   'applied_filters': {'tags':

            ### Filter case: `logical_not_toplevel`

            Top-level logical negation.

            The filter removes the `north` camera-zone document and keeps everything else.

            Expected matching `video_id` values: `['vid-002', 'vid-003', 'vid-004', 'vid-005', 'vid-006']`

            Payload under test:

            ```json
            {
  "where": {
    "not": {
      "field": "camera_zone",
      "op": "eq",
      "value": "north"
    }
  }
}
            ```


In [32]:
case = get_filter_case("logical_not_toplevel")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: logical_not_toplevel

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-002', 'vid-003', 'vid-004', 'vid-005', 'vid-006']`  
**Observed IDs:** `['vid-002', 'vid-003', 'vid-004', 'vid-005', 'vid-006']`

{'request_id': 'd0982db1-1676-4da9-bb28-ae6f0527ad9d',
 'results': [{'query_id': 'logical_not_toplevel',
   'query': 'fixture retrieval anchor',
   'count': 5,
   'items': [{'score': 1.4708623886108398,
     'metadata': {'video_id': 'vid-002',
      'camera_zone': 'south',
      'description': 'blue bus near station',
      'prefix': 'beta-route',
      'frame_number': 20,
      'bucket_name': 'cam-b',
      'tags': ['traffic', 'vehicle', 'bus'],
      'created_at': '2026-03-15T12:00:00+00:00'},
     'page_content': 'blue bus near station'},
    {'score': 1.2737712860107422,
     'metadata': {'video_id': 'vid-006',
      'camera_zone': 'south-west',
      'description': 'empty intersection at night',
      'prefix': 'delta-night',
      'frame_number': 60,
      'bucket_name': 'cam-e',
      'tags': ['night', 'traffic'],
      'created_at': '2026-04-05T20:10:00+00:00'},
     'page_content': 'empty intersection at night'},
    {'score': 1.1559438705444336,
     'metadata': {'video_id': 

            ### Filter case: `no_match`

            Zero-result behavior.

            This case is expected to return an empty `items` list without surfacing an API error.

            Expected matching `video_id` values: `[]`

            Payload under test:

            ```json
            {
  "where": {
    "field": "video_id",
    "op": "eq",
    "value": "vid-999"
  }
}
            ```


In [33]:
case = get_filter_case("no_match")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: no_match

<IPython.core.display.JSON object>

**Expected IDs:** `[]`  
**Observed IDs:** `[]`

{'request_id': 'e04c0b78-50f0-4fc7-af1a-9fc9745e035f',
 'results': [{'query_id': 'no_match',
   'query': 'fixture retrieval anchor',
   'count': 0,
   'items': [],
   'applied_filters': {'tags': None,
    'time_filter': None,
    'filters': None,
    'normalized_where': {'field': 'video_id',
     'op': 'eq',
     'value': 'vid-999',
     'all': None,
     'any': None,
     'not': None},
    'warnings': None,
    'compiled_backend_filter': {'video_id': {'$eq': 'vid-999'}},
    'dropped_or_rewritten_clauses': []}}],
 'errors': []}

            ### Filter case: `op_time_between_where`

            Datetime range filtering expressed directly in `where`.

            The expected matches fall inside the inclusive March 20 to April 1 window.

            Expected matching `video_id` values: `['vid-003', 'vid-004', 'vid-005']`

            Payload under test:

            ```json
            {
  "where": {
    "field": "created_at",
    "op": "between",
    "value": [
      "2026-03-20T00:00:00+00:00",
      "2026-04-01T23:59:59+00:00"
    ]
  }
}
            ```


In [34]:
case = get_filter_case("op_time_between_where")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: op_time_between_where

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-003', 'vid-004', 'vid-005']`  
**Observed IDs:** `['vid-003', 'vid-004', 'vid-005']`

{'request_id': '27a208a6-ddae-4cb4-a7d0-0c346f3fb684',
 'results': [{'query_id': 'op_time_between_where',
   'query': 'fixture retrieval anchor',
   'count': 3,
   'items': [{'score': 1.1559438705444336,
     'metadata': {'video_id': 'vid-003',
      'camera_zone': 'east',
      'description': 'person with bicycle crossing',
      'prefix': 'alpha-bike',
      'frame_number': 30,
      'bucket_name': 'cam-c',
      'tags': ['pedestrian', 'bicycle'],
      'created_at': '2026-03-20T08:30:00+00:00',
      'optional_note': 'pedestrian event'},
     'page_content': 'person with bicycle crossing'},
    {'score': 0.9245074391365051,
     'metadata': {'video_id': 'vid-005',
      'camera_zone': 'north-east',
      'description': 'traffic light malfunction',
      'prefix': 'alpha-signal',
      'frame_number': 50,
      'bucket_name': 'cam-a',
      'tags': ['traffic', 'signal'],
      'created_at': '2026-04-01T09:45:00+00:00',
      'optional_note': 'signal issue'},
     'page_content': 'tra

            ### Filter case: `nested_all`

            Nested `all` blocks.

            This checks that FAISS honors multi-level logical grouping, not only a flat list of clauses.

            Expected matching `video_id` values: `['vid-003']`

            Payload under test:

            ```json
            {
  "where": {
    "all": [
      {
        "all": [
          {
            "field": "frame_number",
            "op": "gte",
            "value": 10
          },
          {
            "field": "frame_number",
            "op": "lte",
            "value": 30
          }
        ]
      },
      {
        "field": "bucket_name",
        "op": "eq",
        "value": "cam-c"
      }
    ]
  }
}
            ```


In [35]:
case = get_filter_case("nested_all")
case_body, case_result = run_filter_case(case)
case_body


POST /query -> 200


### Filter case: nested_all

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

**Expected IDs:** `['vid-003']`  
**Observed IDs:** `['vid-003']`

{'request_id': 'f843dde9-31d5-48d3-b860-cc115e088462',
 'results': [{'query_id': 'nested_all',
   'query': 'fixture retrieval anchor',
   'count': 1,
   'items': [{'score': 1.1559438705444336,
     'metadata': {'video_id': 'vid-003',
      'camera_zone': 'east',
      'description': 'person with bicycle crossing',
      'prefix': 'alpha-bike',
      'frame_number': 30,
      'bucket_name': 'cam-c',
      'tags': ['pedestrian', 'bicycle'],
      'created_at': '2026-03-20T08:30:00+00:00',
      'optional_note': 'pedestrian event'},
     'page_content': 'person with bicycle crossing'}],
   'applied_filters': {'tags': None,
    'time_filter': None,
    'filters': None,
    'normalized_where': {'field': None,
     'op': None,
     'value': None,
     'all': [{'field': None,
       'op': None,
       'value': None,
       'all': [{'field': 'frame_number',
         'op': 'gte',
         'value': 10,
         'all': None,
         'any': None,
         'not': None},
        {'field': 'frame_nu

## Teardown

Run this cell when you are finished exploring. It tears down the compose stack, removes attached volumes for this notebook run, and deletes the repository-local FAISS runtime directory.


In [36]:
cleanup_stack(STACK_ENV)
display(Markdown("Teardown complete."))


$ docker compose -f docker/compose.yaml -f docker/compose.faiss.yaml down -v --remove-orphans


time="2026-05-25T11:13:02+05:30" level=warning msg="The \"REGISTRY\" variable is not set. Defaulting to a blank string."
 Container vrfaissnb6a1e40fb-vector-retriever-1 Stopping 
 Container vrfaissnb6a1e40fb-vector-retriever-1 Stopped 
 Container vrfaissnb6a1e40fb-vector-retriever-1 Removing 
 Container vrfaissnb6a1e40fb-vector-retriever-1 Removed 
 Container multimodal-embedding-serving Stopping 
 Container multimodal-embedding-serving Stopped 
 Container multimodal-embedding-serving Removing 
 Container multimodal-embedding-serving Removed 
 Volume vrfaissnb6a1e40fb_ov-models Removing 
 Volume vrfaissnb6a1e40fb_data-prep Removing 
 Network vrfaissnb6a1e40fb_vr_network Removing 
 Volume vrfaissnb6a1e40fb_ov-models Removed 
 Volume vrfaissnb6a1e40fb_data-prep Removed 
 Network vrfaissnb6a1e40fb_vr_network Removed
Removed runtime directory: /home/intel/workbench/integration/lib/microservices/vector-retriever/vector-retriever/tests/functional/notebooks/.runtime/faiss-functional-6a1e40fb


Teardown complete.

## Cleanup verification

This optional check confirms that the notebook-specific compose project is no longer running and that the runtime directory has been removed.


In [37]:
if STACK_ENV is not None:
    run_compose(["ps"], env=STACK_ENV, check=False)
print({
    "stack_started": NOTEBOOK_STATE["stack_started"],
    "seeded": NOTEBOOK_STATE["seeded"],
    "runtime_dir_exists": NOTEBOOK_RUNTIME_DIR.exists(),
})


$ docker compose -f docker/compose.yaml -f docker/compose.faiss.yaml ps
NAME      IMAGE     COMMAND   SERVICE   CREATED   STATUS    PORTS
time="2026-05-25T11:13:03+05:30" level=warning msg="The \"REGISTRY\" variable is not set. Defaulting to a blank string."
{'stack_started': False, 'seeded': False, 'runtime_dir_exists': False}
